# 项目核心流程演示：HCT116 · single · all（全环境）· CNN(7|3)

本 notebook 用**一组真实交付的配置**把平台核心链路走一遍，只读已经落盘的产物，不重新训练。

| 项 | 取值 |
| :--- | :--- |
| 数据集 | DeepCRISPR（23 nt，8 通道 = 序列 4 + 表观 4） |
| 细胞系 | **hct116**（4,239 条） |
| 划分 | **single**（系内 70/15/15，group-aware，seed 42） |
| 环境组合 | **all**（CTCF + Dnase + H3K4me3 + RRBS 全开） |
| 模型配置 | **CNN(7\|3)**（序列核宽 7、环境核宽 3） |
| run 名 | `single_hct116_cnn_all_kernel_7` |

内容分四节：**① 训练数据 → ② 生成了哪些文件 → ③ 可视化（只看热图）→ ④ 本次跳过的统计处理**。
目录结构遵循 README §13；所有路径相对于仓库根。

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
RUN = ROOT / "results/train_results/DeepCRISPR/single_hct116_cnn_all_kernel_7"
SUM = ROOT / "results/summary/DeepCRISPR"
PROC = ROOT / "data/processed/DeepCRISPR"

print("仓库根   :", ROOT)
print("run 目录 :", RUN.relative_to(ROOT))
print("汇总目录 :", SUM.relative_to(ROOT))
print("配置     : cell_line=hct116 | split_type=single | environment=all | model=cnn(7|3)")
assert RUN.is_dir(), RUN
assert SUM.is_dir(), SUM

仓库根   : /home/zhang/bioprogram/Submit
run 目录 : results/train_results/DeepCRISPR/single_hct116_cnn_all_kernel_7
汇总目录 : results/summary/DeepCRISPR
配置     : cell_line=hct116 | split_type=single | environment=all | model=cnn(7|3)


## ① 训练数据：模型实际读到的张量

训练端**只读 `data/processed/DeepCRISPR/feature_schema.json`** 决定输入形状，不读数据集 JSON。
下面依次是：编码声明 → 实际张量 → 该 run 的划分规模。

In [ ]:
schema = json.loads((PROC / "feature_schema.json").read_text(encoding="utf-8"))
lay = schema["layout"]

print("通道顺序   :", schema["channel_names"])
print("张量形状   :", tuple(lay["tensor_shape"]), "= (序列长度 23, 通道数 8)")
print("展平公式   :", lay["flatten_index_formula"])
print("特征总数   :", schema["feature_count"], "= 23 × 8")
print("序列块     :", lay["sequence_block"])
print("序列编码   :", schema["encoding"]["sequence"]["encoding"],
      schema["encoding"]["sequence"]["alphabet"])
print("环境编码   :", schema["encoding"]["environment"]["default_type"],
      "取值集", schema["encoding"]["environment"]["value_set"])
print("环境通道   :", [f["name"] for f in schema["encoding"]["environment"]["features"]])

通道顺序   : ['A', 'C', 'G', 'T', 'CTCF', 'Dnase', 'H3K4me3', 'RRBS']
张量形状   : (23, 8) = (序列长度 23, 通道数 8)
展平公式   : flat_index = position0_based * channel_count + channel_index
特征总数   : 184 = 23 × 8
序列块     : channel 0..3 为序列；channel 4..7 为环境
序列编码   : one_hot ['A', 'C', 'G', 'T']
环境编码   : per_position_binary 取值集 [0.0, 1.0]
环境通道   : ['CTCF', 'Dnase', 'H3K4me3', 'RRBS']


In [ ]:
X3d = np.load(PROC / "hct116_features_23x8.npy")
X2d = np.load(PROC / "hct116_features_184.npy")
y = np.load(PROC / "hct116_labels.npy")
meta = pd.read_csv(PROC / "hct116_metadata.csv")

print(f"3D 张量 (N, 23, 8) : {X3d.shape}  {X3d.dtype}  取值集合 {np.unique(X3d)}")
print(f"展平张量 (N, 184)  : {X2d.shape}  {X2d.dtype}")
print(f"标签 y             : {y.shape}  min={y.min():.4f}  max={y.max():.4f}  mean={y.mean():.4f}")
print(f"metadata           : {meta.shape}  列={list(meta.columns)}")

share = pd.DataFrame({
    "通道": schema["channel_names"],
    "该通道为 1 的比例": X3d.mean(axis=(0, 1)).round(4),
})
print("\n每个通道的取值密度（序列 4 通道 ≈ 0.25 即 one-hot；环境 4 通道 = 可及性比例）：")
print(share.to_string(index=False))

3D 张量 (N, 23, 8) : (4239, 23, 8)  float32  取值集合 [0. 1.]
展平张量 (N, 184)  : (4239, 184)  float32
标签 y             : (4239,)  min=0.0003  max=0.9459  mean=0.2543
metadata           : (4239, 7)  列=['Cell line', 'Chromosome', 'Start', 'End', 'Strand', 'sgRNA', 'Normalized efficacy']

每个通道的取值密度（序列 4 通道 ≈ 0.25 即 one-hot；环境 4 通道 = 可及性比例）：
     通道  该通道为 1 的比例
      A      0.2386
      C      0.2526
      G      0.3312
      T      0.1776
   CTCF      0.8664
  Dnase      0.9512
H3K4me3      0.8400
   RRBS      0.0093


In [ ]:
cols = ["Cell line", "Chromosome", "Start", "End", "Strand", "sgRNA", "Normalized efficacy"]
print("metadata 前 3 行：")
print(meta[cols].head(3).to_string(index=False))

q = pd.Series(y).quantile([0.0, 0.25, 0.5, 0.75, 1.0]).round(4)
print("\n标签分布（Normalized efficacy，已归一化到 [0,1]）：")
for k, v in q.items():
    print(f"  {int(k*100):>3d}% 分位 = {v}")

metadata 前 3 行：
Cell line Chromosome    Start      End Strand                   sgRNA  Normalized efficacy
   hct116       chr1 19513973 19513995      + ACGTTAGCAGTTTGATGGCATGG             0.158509
   hct116       chr1 38168949 38168971      - ACCTCCAATCGGCCCACGGCTGG             0.147298
   hct116       chr1 44422307 44422329      - CATTGACAGGATAGTGGCCAGGG             0.198014

标签分布（Normalized efficacy，已归一化到 [0,1]）：
    0% 分位 = 0.0003
   25% 分位 = 0.1213
   50% 分位 = 0.2096
   75% 分位 = 0.3601
  100% 分位 = 0.9459


In [ ]:
info = {}
for line in (RUN / "cnn_info.txt").read_text(encoding="utf-8").splitlines():
    if ":" in line:
        k, _, v = line.partition(":")
        info.setdefault(k.strip(), v.strip())

keys = ["run_name", "model", "split_type", "cell_line", "environment", "selected_environments",
        "environment_count", "channel_count", "sequence_kernel", "environment_kernel",
        "random_seed", "n_train", "n_valid", "n_test",
        "input_shape_train", "input_shape_valid", "input_shape_test",
        "audit_train_test_sequence_overlap", "audit_train_test_revcomp_overlap", "split_digest"]
print("该 run 的运行配置与划分审计（摘自 cnn_info.txt）：")
for k in keys:
    print(f"  {k:38s} {info.get(k, '')}")

该 run 的运行配置与划分审计（摘自 cnn_info.txt）：
  run_name                               single_hct116_cnn_all_kernel_7
  model                                  cnn_dual_branch
  split_type                             single
  cell_line                              hct116
  environment                            all
  selected_environments                  ['ctcf', 'dnase', 'h3k4me3', 'rrbs']
  environment_count                      4
  channel_count                          8
  sequence_kernel                        7
  environment_kernel                     3
  random_seed                            42
  n_train                                2968
  n_valid                                635
  n_test                                 636
  input_shape_train                      [2968, 23, 8]
  input_shape_valid                      [635, 23, 8]
  input_shape_test                       [636, 23, 8]
  audit_train_test_sequence_overlap      0
  audit_train_test_revcomp_overlap       0
  split_digest  

## ② 生成了哪些文件

一次 run 固定落 7 个文件；批次级汇总另有 `reports/ tables/ train_data/ feature_importance/ figures/` 五类。

In [ ]:
rows = []
for p in sorted(RUN.iterdir()):
    rows.append({"文件": p.name, "大小": f"{p.stat().st_size / 1024:.1f} KB"})
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nrun 目录合计：{sum(p.stat().st_size for p in RUN.iterdir()) / 1024:.1f} KB")

                            文件      大小
    cnn_feature_importance.csv 13.9 KB
                  cnn_info.txt  2.1 KB
              cnn_metrics.json  0.2 KB
           cnn_predictions.csv 20.8 KB
      cnn_training_history.csv  1.3 KB
   cnn_validation_metrics.json  0.2 KB
cnn_validation_predictions.csv 20.8 KB

run 目录合计：59.3 KB


In [ ]:
test = json.loads((RUN / "cnn_metrics.json").read_text(encoding="utf-8"))
val = json.loads((RUN / "cnn_validation_metrics.json").read_text(encoding="utf-8"))
order = ["R2", "RMSE", "MAE", "Pearson", "Spearman", "MSE"]
print(pd.DataFrame({"测试集": pd.Series(test), "验证集": pd.Series(val)}).loc[order].round(4).to_string())
print("\n口径：*_metrics.json = 测试集（唯一性能口径）；*_validation_metrics.json = 验证集（仅用于早停选 epoch）。")

             测试集     验证集
R2        0.1432  0.1288
RMSE      0.1669  0.1594
MAE       0.1296  0.1250
Pearson   0.3866  0.3689
Spearman  0.3734  0.3632
MSE       0.0278  0.0254

口径：*_metrics.json = 测试集（唯一性能口径）；*_validation_metrics.json = 验证集（仅用于早停选 epoch）。


In [ ]:
fi = pd.read_csv(RUN / "cnn_feature_importance.csv")
print(f"本 run 的归因表 cnn_feature_importance.csv：{fi.shape[0]} 行 × {fi.shape[1]} 列")
print("列（XAI 白名单，见 README §17.0）:", list(fi.columns))
print(fi.head(5).round(6).to_string(index=False))

本 run 的归因表 cnn_feature_importance.csv：184 行 × 6 列
列（XAI 白名单，见 README §17.0）: ['Feature', 'Position', 'Channel', 'CNN_IG', 'CNN_ISM', 'ISM_SNR']
 Feature  Position Channel   CNN_IG  CNN_ISM  ISM_SNR
C_pos_17        17       C 0.014193 0.055655 1.957786
A_pos_22        22       A 0.000000 0.039859 1.687518
A_pos_17        17       A 0.020158 0.039174 1.610260
A_pos_21        21       A 0.000000 0.038001 1.640351
A_pos_18        18       A 0.016909 0.035587 1.497171


In [ ]:
print("批次级总入口（analysis.pipeline）产出的五类目录：")
for sub in ["reports", "tables", "train_data", "feature_importance", "figures"]:
    d = SUM / sub
    n = sum(1 for p in d.rglob("*") if p.is_file()) if d.is_dir() else 0
    print(f"  {sub:20s} {n:4d} 个文件")

print("\ntables/ 全清单（20 张）：")
print("  " + "\n  ".join(sorted(p.name for p in (SUM / "tables").glob("*.csv"))))
print("\ntrain_data/ 全清单（5 张）：")
print("  " + "\n  ".join(sorted(p.name for p in (SUM / "train_data").glob("*.csv"))))
print("\nfeature_importance/ 全清单（8 个）：")
print("  " + "\n  ".join(sorted(p.name for p in (SUM / "feature_importance").iterdir())))

批次级总入口（analysis.pipeline）产出的五类目录：
  reports                 3 个文件
  tables                 20 个文件
  train_data              5 个文件
  feature_importance      8 个文件
  figures               198 个文件

tables/ 全清单（20 张）：
  anomaly_report.csv
  anova_results.csv
  attribution_summary.csv
  bootstrap_cellline_effects.csv
  bootstrap_main_effects.csv
  bootstrap_results.csv
  environment_conditional_delta_r2.csv
  environment_dag_report.csv
  environment_edges.csv
  environment_interactions.csv
  environment_main_effects.csv
  environment_nodes.csv
  experiment_table.csv
  loco_performance.csv
  metric_inconsistency.csv
  motif_candidates.csv
  motif_enrichment.csv
  motif_instances.csv
  permutation_results.csv
  prediction_summary.csv

train_data/ 全清单（5 张）：
  all_cell_line_result.csv
  all_experiments.csv
  baseline.csv
  mixed_cell_line_result.csv
  single_cell_line_result.csv

feature_importance/ 全清单（8 个）：
  cnn33_importance.md
  cnn53_importance.md
  cnn73_importance.md
  key_regulatory_bio

In [ ]:
st = json.loads((SUM / "analysis_status.json").read_text(encoding="utf-8"))
print(f"analysis_status.json：{len(st['tasks'])} 个任务")
print(pd.DataFrame([{k: t[k] for k in ("task_id", "selected", "available", "status")}
                    for t in st["tasks"]]).to_string(index=False))

analysis_status.json：13 个任务
                       task_id  selected  available    status
                            qc      True       True completed
                    prediction      True       True completed
environment_conditional_effect      True       True completed
       environment_main_effect      True       True completed
     environment_factorial_dag      True       True completed
          sequence_attribution      True       True completed
                       cnn_ism      True       True completed
               motif_discovery      True       True completed
              motif_enrichment      True       True completed
                     bootstrap      True       True completed
            hypothesis_testing      True       True completed
                fdr_correction      True       True completed
             environment_anova      True       True completed


## ③ 可视化（只看热图）

图片是**批量级**产物（`results/summary/DeepCRISPR/figures/`），与本节配置直接对应的是：

| 图 | 路径 | 含义 |
| :--- | :--- | :--- |
| 条件 ΔR² 热图 | `figures/environment/conditional_delta_r2_heatmap.png` | 行 = 新加入的环境因子 e，列 = 背景组合 S，格子 = 条件 ΔR² |
| 位置归因热图（CNN ISM） | `figures/plots/position_attribution_cnn_ism.png` | 位置(23) × 通道(8) 的 ISM 归因幅度 |
| 位置归因热图（CNN IG） | `figures/plots/position_attribution_cnn_ig.png` | 同一张表的 IG 归因幅度 |

In [ ]:
figs = [
    "figures/environment/conditional_delta_r2_heatmap.png",
    "figures/plots/position_attribution_cnn_ism.png",
    "figures/plots/position_attribution_cnn_ig.png",
]
for rel in figs:
    p = SUM / rel
    print(f"{'存在' if p.is_file() else '缺失'}  {rel:52s} {p.stat().st_size / 1024:6.0f} KB")

存在  figures/environment/conditional_delta_r2_heatmap.png    133 KB
存在  figures/plots/position_attribution_cnn_ism.png           60 KB
存在  figures/plots/position_attribution_cnn_ig.png            56 KB


![条件 ΔR² 热图](../results/summary/DeepCRISPR/figures/environment/conditional_delta_r2_heatmap.png)

![位置归因热图（CNN ISM）](../results/summary/DeepCRISPR/figures/plots/position_attribution_cnn_ism.png)

![位置归因热图（CNN IG）](../results/summary/DeepCRISPR/figures/plots/position_attribution_cnn_ig.png)

In [ ]:
# 热图的底层数据在这里（数值可直接核对）
cond = pd.read_csv(SUM / "tables/environment_conditional_delta_r2.csv", encoding="utf-8-sig")
at = pd.read_csv(SUM / "tables/attribution_summary.csv", encoding="utf-8-sig", low_memory=False)

print(f"conditional_delta_r2.csv : {cond.shape[0]} 行 × {cond.shape[1]} 列")
print(f"attribution_summary.csv  : {at.shape[0]} 行 × {at.shape[1]} 列")
print("\n条件 ΔR²（该 run 的上下文，single / hct116 / cnn(7|3)）前 5 行：")
sub = cond[(cond.split_type == "single") & (cond.cell_line == "hct116") & (cond.model == "cnn(7|3)")]
print(sub[["environment_added", "background", "delta_r2_mean", "n_paired"]].head(5).to_string(index=False))
print("\n位置归因（CNN ISM，前 5 行）：")
print(at[at.method == "cnn_ism"][["feature", "channel", "position", "importance", "snr"]].head(5).round(6).to_string(index=False))

conditional_delta_r2.csv : 2016 行 × 10 列
attribution_summary.csv  : 331200 行 × 14 列

条件 ΔR²（该 run 的上下文，single / hct116 / cnn(7|3)）前 5 行：
environment_added                  background  delta_r2_mean  n_paired
             ctcf                    sequence       0.028611         1
             ctcf              sequence_dnase      -0.004659         1
             ctcf      sequence_dnase_h3k4me3       0.015788         1
             ctcf sequence_dnase_h3k4me3_rrbs      -0.008083         1
             ctcf         sequence_dnase_rrbs      -0.005720         1

位置归因（CNN ISM，前 5 行）：
 feature channel  position  importance      snr
 C_pos_0       C         1    0.017252 1.286926
T_pos_19       T        20    0.016730 0.964912
G_pos_22       G        23    0.016297 1.024769
A_pos_20       A        21    0.016262 1.082017
C_pos_19       C        20    0.015840 0.948874


## ④ 本次跳过的统计处理

以下环节需要**跨上下文 / 跨细胞系 / 跨 seed** 才成立，在"单细胞系 + 单 split"的演示配置下不展开
（不是平台做不到，而是这一组配置的数据量不足以支撑）：

| 跳过项 | 本次不展示的原因 | 需要什么才成立 |
| :--- | :--- | :--- |
| 析因方差分析（Type-II 边际 ANOVA） | 可用性守卫要求 `n_obs ≥ min_observations = 32` 且 `df_den ≥ min_residual_df = 5`；单细胞系 + 单 split 的上下文行数不够 | 多 `cell_line` × 多 `split_type` 的合并设计 |
| 两因子交互项 `I(A,B)` | 交互需要"背景含 b"与"背景不含 b"两侧都有配对观测；单 seed 下背景层不可配对 | 跨 seed / 跨细胞系的背景对比 |
| 跨细胞系泛化 `cellline_ratio` | 单细胞系时 `total_celllines = 1`，比例判据没有区分度 | ≥ 2 个细胞系 |
| 全量 336 单元网格训练 | 演示只固定一组配置 | `data_digging.py` 展开完整网格（7 配置 × 3 划分 × 16 环境组合） |

> 这些产物在交付批次里都是存在的，可直接打开核对：
> `tables/anova_results.csv`（94 行）、`tables/environment_interactions.csv`（378 行）、
> `results/赛道二_results/DeepCRISPR/csv/microenv_and_motif.csv`（跨细胞系泛化）。
> 本配置对应的析因图 `figures/environment/factorial_dag/single_hct116_cnn_7_3.png` 也已生成，
> 但按本节范围不在此展开。

## ⑤ 复现本 notebook 的全部命令

下面是这一组配置从原始 CSV 到报告/表格/图的完整命令（与 README §14 一致）：

In [ ]:
cmds = [
    "# ① 预处理：原始 CSV → 张量 + feature_schema.json",
    "python core/features/engineering/feature_engineering.py \\",
    "    --raw-data   data/raw/DeepCRISPR \\",
    "    --output-dir data/processed/DeepCRISPR \\",
    "    --config     data/config/DeepCRISPR.json",
    "",
    "# ② 单 run 训练（本 notebook 对应的那一组：hct116 / single / all / cnn(7|3)）",
    "python train.py --model cnn --split-type single --cell-line hct116 \\",
    "    --environment all --sequence-kernel 7 --environment-kernel 3 \\",
    "    --data-set DeepCRISPR \\",
    "    --results-dir results/train_results/DeepCRISPR \\",
    "    --model-dir   models/DeepCRISPR \\",
    "    --logs-dir    logs/DeepCRISPR \\",
    "    --run-name single_hct116_cnn_all_kernel_7 --seed 42",
    "",
    "# ③ 结果汇总 + 关键调控特征库",
    "python -m analysis.collect_results --batch-dir results/summary/DeepCRISPR",
    "python analysis/importance_extraction.py --batch_dir results/summary/DeepCRISPR",
    "",
    "# ④ 报告 / 表格 / 图（就地写回交付目录）",
    "python -m analysis.pipeline --batch-dir results/summary/DeepCRISPR",
]
print("\n".join(cmds))

# ① 预处理：原始 CSV → 张量 + feature_schema.json
python core/features/engineering/feature_engineering.py \
    --raw-data   data/raw/DeepCRISPR \
    --output-dir data/processed/DeepCRISPR \
    --config     data/config/DeepCRISPR.json

# ② 单 run 训练（本 notebook 对应的那一组：hct116 / single / all / cnn(7|3)）
python train.py --model cnn --split-type single --cell-line hct116 \
    --environment all --sequence-kernel 7 --environment-kernel 3 \
    --data-set DeepCRISPR \
    --results-dir results/train_results/DeepCRISPR \
    --model-dir   models/DeepCRISPR \
    --logs-dir    logs/DeepCRISPR \
    --run-name single_hct116_cnn_all_kernel_7 --seed 42

# ③ 结果汇总 + 关键调控特征库
python -m analysis.collect_results --batch-dir results/summary/DeepCRISPR
python analysis/importance_extraction.py --batch_dir results/summary/DeepCRISPR

# ④ 报告 / 表格 / 图（就地写回交付目录）
python -m analysis.pipeline --batch-dir results/summary/DeepCRISPR


---

**口径提示**

- 本 notebook 里所有数值都来自已落盘产物；`*_metrics.json` 才是性能口径，`*_validation_metrics.json`
  只用于早停选 epoch。
- `attribution_summary.csv` 的 `feature` 列有**两套位置编号并存**（`<ch>_pos_<0-based>` 与
  `pos<1-based>_<ch>`）；跨方法比较请用规范化的 `position` 列或 `(channel, position)`，不要 join `feature`。
- 环境（表观）通道的贡献只在 DeepCRISPR 上可算；纯序列数据集（Hiranniramol / Labuhn）没有这一维度。